In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Embedding
from torch.utils.data import DataLoader
from datasets import load_dataset
from tqdm.auto import tqdm

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

device

In [ ]:
ds = load_dataset("imdb")
ds

In [ ]:
import re
from collections import Counter

def simple_tokenize(text: str):
    text = text.lower()
    text = re.sub(r"<br\s*/?>", " ", text)
    # keep words and basic punctuation as separate tokens
    tokens = re.findall(r"[a-z0-9']+|[.,!?;]", text)
    return tokens

# Build vocab from training set (optionally limit size)
MAX_VOCAB = 20000
MIN_FREQ = 2
PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"

counter = Counter()
for ex in tqdm(ds["train"], desc="Building vocab"):
    counter.update(simple_tokenize(ex["text"]))

# keep tokens with freq >= MIN_FREQ
tokens_and_freqs = [(t, f) for t, f in counter.items() if f >= MIN_FREQ]
tokens_and_freqs.sort(key=lambda x: x[1], reverse=True)

vocab_tokens = [PAD_TOKEN, UNK_TOKEN] + [t for t, _ in tokens_and_freqs[:MAX_VOCAB-2]]
stoi = {t: i for i, t in enumerate(vocab_tokens)}
itos = vocab_tokens

pad_id = stoi[PAD_TOKEN]
unk_id = stoi[UNK_TOKEN]

len(itos), itos[:20]

In [ ]:
MAX_LEN = 256  # truncate long reviews

def encode(text: str):
    toks = simple_tokenize(text)[:MAX_LEN]
    ids = [stoi.get(t, unk_id) for t in toks]
    return ids

def collate_batch(batch):
    # batch is a list of dicts with keys: "text", "label"
    sequences = [encode(x["text"]) for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)

    lengths = torch.tensor([len(seq) for seq in sequences], dtype=torch.long)
    max_len = lengths.max().item() if len(lengths) > 0 else 0

    padded = torch.full((len(sequences), max_len), pad_id, dtype=torch.long)
    for i, seq in enumerate(sequences):
        if len(seq) > 0:
            padded[i, :len(seq)] = torch.tensor(seq, dtype=torch.long)

    return padded, lengths, labels

In [ ]:
BATCH_SIZE = 64

train_loader = DataLoader(ds["train"], batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
test_loader  = DataLoader(ds["test"],  batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)

In [ ]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)
        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,   # <- unidirectional
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x, lengths):
        # x: (B, T)
        emb = self.embedding(x)  # (B, T, E)

        # Pack padded sequence
        lengths_cpu = lengths.cpu()
        packed = nn.utils.rnn.pack_padded_sequence(emb, lengths_cpu, batch_first=True, enforce_sorted=False)
        packed_out, (h_n, c_n) = self.lstm(packed)

        # h_n: (num_layers * num_directions, B, hidden_dim)
        last_hidden = h_n[-1]  # (B, hidden_dim) since unidirectional
        out = self.dropout(last_hidden)
        logits = self.fc(out).squeeze(1)  # (B,)
        return logits

In [ ]:
class BiSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        # hidden_dim * 2 because of bidirectionality
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, lengths):
        # x: (B, T)
        emb = self.embedding(x)  # (B, T, E)

        lengths_cpu = lengths.cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths_cpu, batch_first=True, enforce_sorted=False
        )

        _, (h_n, _) = self.lstm(packed)

        # h_n shape:
        # (num_layers * 2, B, hidden_dim)
        # last layer:
        #   h_n[-2] -> forward
        #   h_n[-1] -> backward

        forward_hidden  = h_n[-2]
        backward_hidden = h_n[-1]

        # Concatenate forward + backward
        hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        hidden = self.dropout(hidden)

        logits = self.fc(hidden).squeeze(1)
        return logits

In [ ]:
import zipfile
import urllib
import os


class GloveSentimentLSTM(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding, emb_dim = self._load_glove()
        # self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,     # <<< CHANGE
            dropout=dropout if num_layers > 1 else 0.0
        )

        self.dropout = nn.Dropout(dropout)

        # hidden_dim * 2 because of bidirectionality
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x, lengths):
        # x: (B, T)
        emb = self.embedding(x)  # (B, T, E)

        lengths_cpu = lengths.cpu()
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths_cpu, batch_first=True, enforce_sorted=False
        )

        _, (h_n, _) = self.lstm(packed)

        # h_n shape:
        # (num_layers * 2, B, hidden_dim)
        # last layer:
        #   h_n[-2] -> forward
        #   h_n[-1] -> backward

        forward_hidden  = h_n[-2]
        backward_hidden = h_n[-1]

        # Concatenate forward + backward
        hidden = torch.cat((forward_hidden, backward_hidden), dim=1)
        hidden = self.dropout(hidden)

        logits = self.fc(hidden).squeeze(1)
        return logits

    def _load_glove(self):
        # Directory for GloVe
        glove_dir = "glove"
        zip_path = os.path.join(glove_dir, "glove.6B.zip")

        # Files we expect after extraction
        expected_files = [
            "glove.6B.50d.txt",
            "glove.6B.100d.txt",
            "glove.6B.200d.txt",
            "glove.6B.300d.txt",
        ]

        # 1. Create directory if missing
        os.makedirs(glove_dir, exist_ok=True)

        # 2. Check if the ZIP file exists
        if not os.path.exists(zip_path):
            print("GloVe ZIP not found. Downloading…")
            url = "https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip"
            urllib.request.urlretrieve(url, zip_path)
            print("Download complete.")
        else:
            print("ZIP file already exists — skipping download.")

        # 3. Check if files are already extracted
        if not all(os.path.exists(os.path.join(glove_dir, f)) for f in expected_files):
            print("Extracting GloVe files…")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(glove_dir)
            print("Extraction complete.")
        else:
            print("GloVe text files already extracted — skipping extraction.")
        glove50 = {}
        with open("glove/glove.6B.50d.txt", "r", encoding="utf8") as f:
            for line in f:
                parts = line.strip().split()
                word = parts[0]
                vector = np.array(parts[1:], dtype=float)
                glove50[word] = vector

        # Build a matrix and index mapping
        vocab = list(glove50.keys())

        embedding_matrix = torch.tensor(
            np.stack([glove50[w] for w in vocab]),
            dtype=torch.float
        )
        embedding = nn.Embedding.from_pretrained(embedding_matrix, freeze=False)

        return embedding, 50

# Attention Layer after a BiLSTM (Classic RNN Attention)

**classic attention mechanism** applied **after a bidirectional LSTM and before a final classifier**, as used in sentiment analysis
(e.g. IMDB).

This is **not Transformer attention**, but an RNN-style additive attention inspired by
Bahdanau et al.

---

## 1. Context: Output of the BiLSTM

After a bidirectional LSTM processes a batch of padded sequences, it produces:

- One hidden state per token
- Each hidden state encodes **past and future context**

Let:
- $B$ = batch size
- $T$ = sequence length (after padding)
- $H$ = hidden size per direction

The output tensor has shape:

$$
\text{outputs} \in \mathbb{R}^{B \times T \times 2H}
$$

Each vector $h_t \in \mathbb{R}^{2H}$ represents token $t$ in context.

---

## 2. Goal of the Attention Layer

The goal of attention is to convert a **sequence of vectors** into a **single vector**
that best summarizes the sequence for the task.

Instead of:
- taking the last hidden state, or
- averaging all states,

attention computes a **weighted sum**:

$$
c = \sum_t \alpha_t h_t
$$

Where:
- $\alpha_t$ is the attention weight for token $t$
- $\sum_t \alpha_t = 1$

The resulting vector $c$ is called the **context vector**.

---

## 3. Step 1 — Compute Attention Scores

Each LSTM output $h_t$ is mapped to a **scalar importance score**.

### Additive (Bahdanau-style) scoring

For each time step $t$:

$$
u_t = \tanh(W h_t + b)
$$

$$
e_t = v^\top u_t
$$

Where:
- $h_t \in \mathbb{R}^{2H}$ is the BiLSTM output
- $W \in \mathbb{R}^{2H \times 2H}$ is a learnable projection matrix
- $v \in \mathbb{R}^{2H}$ is a learnable vector
- $e_t \in \mathbb{R}$ is the raw attention score

Example implementation (PyTorch):

```python
scores = self.v(torch.tanh(self.proj(outputs))).squeeze(-1)
```

Tensor shapes:
- outputs: $(B, T, 2H)$
- after projection + tanh: $(B, T, 2H)$
- after $v$: $(B, T)$

**Intuition**

The attention layer learns what an *informative token representation* looks like for the task.
Tokens that resemble this pattern receive higher scores.

---

## 4. Step 2 — Mask Padding Tokens

Because sequences are padded, not all time steps correspond to real tokens.

A boolean mask is created:
- True  → valid token
- False → padding

Padding scores are suppressed:

- valid token: $e_t = e_t$
- padding token: $e_t = -\infty$

Example:

```python
scores = scores.masked_fill(~mask, float("-inf"))
```

**Why this works**

When softmax is applied, $\exp(-\infty) = 0$, so padding tokens receive **zero attention weight**.

---

## 5. Step 3 — Softmax to Obtain Attention Weights

The masked scores are normalized across time:

$$
\alpha_t = \frac{\exp(e_t)}{\sum_k \exp(e_k)}
$$

Example:

```python
alpha = torch.softmax(scores, dim=1)
```

Properties:
- $0 \le \alpha_t \le 1$
- $\sum_t \alpha_t = 1$

The model is deciding **where to focus** in the sequence.

---

## 6. Step 4 — Compute the Context Vector

The final sequence representation is a weighted sum of LSTM outputs:

$$
c = \sum_t \alpha_t h_t
$$

Example:

```python
context = torch.bmm(alpha.unsqueeze(1), outputs).squeeze(1)
```

Tensor shapes:
- alpha.unsqueeze(1): $(B, 1, T)$
- outputs: $(B, T, 2H)$
- context: $(B, 2H)$

This context vector summarizes the entire sequence.

---

## 7. Step 5 — Classification

The context vector is passed to a classifier:

$$
\text{logits} = W_c c + b_c
$$

Example:

```python
logits = self.fc(context)
```

A sigmoid is applied afterwards to obtain the sentiment probability.

---

## 8. Why Attention Helps

Compared to using only the last hidden state:

- Focuses on **sentiment-bearing words**
- Handles long sequences better
- Provides **interpretability**

Typical high-attention words in IMDB:
- not
- but
- excellent
- boring
- waste

---

## 9. Relation to Classic RNN Attention

This mechanism follows the same principles as:
- Bahdanau (additive) attention
- Luong attention (conceptually)

Differences from Transformer attention:
- Single-head
- No query–key–value decomposition
- No self-attention matrix
- Sequential (RNN-based)

It is best understood as **learned sequence pooling**.

---

## 10. Output of the Attention Layer

The attention layer produces:
- a context vector (used for classification)
- attention weights $\alpha_t$ (useful for visualization)

This supports:
- interpretability
- qualitative analysis
- teaching demonstrations

---

## 11. Summary

The attention mechanism:
- assigns relevance scores to tokens
- ignores padding via masking
- normalizes scores using softmax
- produces a single informative vector

It improves both **performance** and **interpretability**, making it a natural extension
of BiLSTM-based sentiment models.


In [ ]:
class AdditiveAttention(nn.Module):
    """
    Classic (Bahdanau-style) additive attention over encoder outputs.
    Returns a single context vector per sequence: (B, H)
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, outputs, mask):
        """
        outputs: (B, T, H)
        mask:    (B, T)  with True for valid tokens, False for padding
        """
        # (B, T, H) -> (B, T, H) -> (B, T, 1) -> (B, T)
        scores = self.v(torch.tanh(self.proj(outputs))).squeeze(-1)

        # mask out padding positions
        scores = scores.masked_fill(~mask, float("-inf"))

        # attention weights
        alpha = torch.softmax(scores, dim=1)  # (B, T)

        # weighted sum of outputs -> context
        context = torch.bmm(alpha.unsqueeze(1), outputs).squeeze(1)  # (B, H)
        return context, alpha


class AttentionSentimentLSTM(nn.Module):
    def __init__(self, vocab_size, emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_id)

        self.lstm = nn.LSTM(
            input_size=emb_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,  # BiLSTM
            dropout=dropout if num_layers > 1 else 0.0
        )

        lstm_out_dim = hidden_dim * 2  # because bidirectional
        self.attn = AdditiveAttention(lstm_out_dim)

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(lstm_out_dim, 1)

    def forward(self, x, lengths):
        """
        x: (B, T)
        lengths: (B,)
        """
        emb = self.embedding(x)  # (B, T, E)

        # Pack -> LSTM -> Unpack (to get per-timestep outputs)
        packed = nn.utils.rnn.pack_padded_sequence(
            emb, lengths.cpu(), batch_first=True, enforce_sorted=False
        )
        packed_out, _ = self.lstm(packed)
        outputs, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)  # (B, T, 2H)

        # Build mask for valid timesteps (True where index < length)
        B, T, _ = outputs.size()
        idx = torch.arange(T, device=lengths.device).unsqueeze(0).expand(B, T)
        mask = idx < lengths.unsqueeze(1)  # (B, T) boolean

        # Attention over outputs -> context vector
        context, alpha = self.attn(outputs, mask)  # context: (B, 2H)

        context = self.dropout(context)
        logits = self.fc(context).squeeze(1)  # (B,)
        return logits

In [ ]:
model = AttentionSentimentLSTM(vocab_size=len(itos), emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(device)

# model = GloveSentimentLSTM(hidden_dim=128, num_layers=1, dropout=0.3).to(device)
# model = BiSentimentLSTM(vocab_size=len(itos), emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(device)
# model = SentimentLSTM(vocab_size=len(itos), emb_dim=128, hidden_dim=128, num_layers=1, dropout=0.3).to(device)
model

In [ ]:
def accuracy_from_logits(logits, labels):
    # logits: real numbers, labels: 0/1
    probs = torch.sigmoid(logits)
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Train", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Eval", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

In [ ]:
def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Train", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(x, lengths)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits.detach(), y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total_acc = 0.0
    n_batches = 0

    for x, lengths, y in tqdm(loader, desc="Eval", leave=False):
        x, lengths, y = x.to(device), lengths.to(device), y.to(device)
        logits = model(x, lengths)
        loss = criterion(logits, y)

        total_loss += loss.item()
        total_acc += accuracy_from_logits(logits, y)
        n_batches += 1

    return total_loss / n_batches, total_acc / n_batches

In [ ]:
EPOCHS = 5

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader)
    test_loss, test_acc = evaluate(model, test_loader)
    print(f"Epoch {epoch:02d} | train loss {train_loss:.4f} acc {train_acc:.4f} | test loss {test_loss:.4f} acc {test_acc:.4f}")

In [ ]:
@torch.no_grad()
def predict_sentiment(text: str):
    model.eval()
    ids = encode(text)
    x = torch.tensor(ids, dtype=torch.long).unsqueeze(0).to(device)
    lengths = torch.tensor([len(ids)], dtype=torch.long).to(device)
    logits = model(x, lengths)
    prob = torch.sigmoid(logits).item()
    return prob

texts = [
    "This movie was fantastic, I loved it!",
    "Boring and too long. I regret watching it."
]
for t in texts:
    p = predict_sentiment(t)
    print(f"{p:.3f} -> {'POS' if p>=0.5 else 'NEG'} | {t}")